# Hydrogeological and Geospatial Data for Surface and Subsurface Flow Modelling in Swedish Coastal Catchments Exploration with `mlcroissant`
This notebook provides a guided workflow for loading, inspecting, and exploring a Croissant-described dataset using the `mlcroissant` library.

The dataset used here is a FAIR² hydrogeological and geospatial dataset for a Swedish coastal catchment. We will:
- load the dataset metadata from the Croissant schema URL,
- inspect available record sets and fields using their `@id` values,
- load records into pandas DataFrames,
- run a lightweight exploratory analysis, and
- generate a few visualizations when tabular numeric data are available.

### Dataset Source
Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.8kx9-mx2j/fair2.json`

> Important: throughout this notebook, record sets, fields, and related entities are referenced by their `@id` values whenever available.

In [ ]:
# Ensure `mlcroissant` and supporting libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn numpy

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

This section keeps the Croissant URL in a variable so the notebook remains easy to adapt to another dataset.

In [ ]:
import json
import math
import warnings

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.8kx9-mx2j/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully.")
print(f"Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description preview: {str(getattr(metadata, 'description', 'N/A'))[:500]}...")

In [ ]:
# Convert metadata to JSON for easier inspection while keeping dataset.metadata as a single object.
metadata_json = metadata.to_json()

summary = {
    "@id": metadata_json.get("@id"),
    "name": metadata_json.get("name"),
    "identifier": metadata_json.get("identifier"),
    "version": metadata_json.get("version"),
    "license": metadata_json.get("license"),
    "citeAs": metadata_json.get("citeAs"),
    "spatialCoverage": metadata_json.get("spatialCoverage"),
    "datePublished": metadata_json.get("datePublished")
}

pd.DataFrame([summary]).T.rename(columns={0: "value"})

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets can expose one or more record sets. In many workflows, the first task is to inspect the schema and list the available `@id` values for record sets and fields.

The helper code below builds a compact overview table from the Croissant metadata JSON.

In [ ]:
def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

record_sets_json = as_list(metadata_json.get("recordSet"))
print(f"Number of record sets declared in metadata: {len(record_sets_json)}")
record_sets_json

In [ ]:
# Build an overview table for record sets and fields using their @id values.
record_set_rows = []
field_rows = []

for rs in record_sets_json:
    rs_id = rs.get("@id", "N/A") if isinstance(rs, dict) else "N/A"
    rs_name = rs.get("name", "N/A") if isinstance(rs, dict) else "N/A"
    rs_desc = rs.get("description", "") if isinstance(rs, dict) else ""
    fields = as_list(rs.get("field")) if isinstance(rs, dict) else []
    record_set_rows.append({
        "record_set_@id": rs_id,
        "record_set_name": rs_name,
        "field_count": len(fields),
        "description": str(rs_desc)[:150]
    })
    
    for fld in fields:
        if isinstance(fld, dict):
            field_rows.append({
                "record_set_@id": rs_id,
                "field_@id": fld.get("@id", "N/A"),
                "field_name": fld.get("name", "N/A"),
                "dataType": fld.get("dataType", "N/A"),
                "source": fld.get("source", "N/A")
            })

record_sets_df = pd.DataFrame(record_set_rows)
fields_df = pd.DataFrame(field_rows)

print("Record set overview:")
display(record_sets_df if not record_sets_df.empty else pd.DataFrame([{"message": "No record sets were declared in metadata."}]))

print("Field overview:")
display(fields_df.head(50) if not fields_df.empty else pd.DataFrame([{"message": "No fields were declared in metadata."}]))

The template notebook includes a direct example of iterating through records from a chosen record set. Since this dataset may expose zero or more record sets depending on the package structure, the next cell dynamically selects the first available record set `@id` when present.

In [ ]:
record_set_ids = record_sets_df["record_set_@id"].tolist() if not record_sets_df.empty else []
selected_record_set_id = record_set_ids[0] if record_set_ids else None

print(f"Selected record set @id: {selected_record_set_id}")

if selected_record_set_id is not None:
    print("Sample records from the selected record set:")
    for i, x in enumerate(dataset.records(record_set=selected_record_set_id)):
        print(x)
        if i >= 4:
            break
else:
    print("No record set @id is available in the metadata, so no records can be iterated in this step.")

## 3. Data Extraction
Load data from each available record set into pandas DataFrames for analysis.

As requested, the workflow below uses record set `@id` values dynamically. If no record sets are available, the notebook still remains informative by working from the metadata summary.

In [ ]:
# Extract data from each record set using record set @id values
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

print(f"Total loaded DataFrames: {len(dataframes)}")

In [ ]:
if dataframes:
    first_loaded_record_set_id = list(dataframes.keys())[0]
    print(f"First loaded record set @id: {first_loaded_record_set_id}")
    print("Columns:")
    print(dataframes[first_loaded_record_set_id].columns.tolist())
    display(dataframes[first_loaded_record_set_id].head())
else:
    print("No record-set tables were loaded. This can happen if the Croissant package currently exposes metadata and file distributions without explicit recordSet entries.")

Because this dataset package appears to emphasize metadata, distributions, and documentation, it is also useful to inspect the dataset distributions directly. Each distribution is referenced by its `@id`.

In [ ]:
distribution_rows = []
for dist in as_list(metadata_json.get("distribution")):
    if isinstance(dist, dict):
        distribution_rows.append({
            "distribution_@id": dist.get("@id"),
            "name": dist.get("name", "N/A"),
            "contentUrl": dist.get("contentUrl", "N/A"),
            "encodingFormat": dist.get("encodingFormat", "N/A"),
            "description": str(dist.get("description", ""))[:120]
        })
    else:
        distribution_rows.append({
            "distribution_@id": str(dist),
            "name": "N/A",
            "contentUrl": "N/A",
            "encodingFormat": "N/A",
            "description": ""
        })

distributions_df = pd.DataFrame(distribution_rows)
display(distributions_df.head(50))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section is written defensively so it can handle either:
1. one or more loaded record sets with tabular data, or
2. a metadata-focused package with no explicit record sets.

When tabular record sets are available, we automatically choose a numeric field and a grouping field. Otherwise, we create a compact metadata-based analysis table from dataset-level descriptive lists such as keywords, use cases, and limitations.

In [ ]:
analysis_df = None
analysis_record_set_id = None
numeric_field_id = None
group_field_id = None

if dataframes:
    # Choose the first non-empty dataframe
    for rs_id, df in dataframes.items():
        if not df.empty:
            analysis_df = df.copy()
            analysis_record_set_id = rs_id
            break

if analysis_df is not None:
    numeric_candidates = analysis_df.select_dtypes(include=[np.number]).columns.tolist()
    object_candidates = analysis_df.select_dtypes(include=["object", "category"]).columns.tolist()

    numeric_field_id = numeric_candidates[0] if numeric_candidates else None
    group_field_id = object_candidates[0] if object_candidates else None

    print(f"Using record set @id for analysis: {analysis_record_set_id}")
    print(f"Selected numeric field @id/column: {numeric_field_id}")
    print(f"Selected group field @id/column: {group_field_id}")
else:
    # Metadata-based fallback table
    metadata_analysis_rows = [
        {"category_@id": "keywords", "value": item, "value_length": len(str(item))}
        for item in as_list(metadata_json.get("keywords"))
    ]
    metadata_analysis_rows += [
        {"category_@id": "dataUseCases", "value": item, "value_length": len(str(item))}
        for item in as_list(metadata_json.get("dataUseCases"))
    ]
    metadata_analysis_rows += [
        {"category_@id": "dataLimitations", "value": item, "value_length": len(str(item))}
        for item in as_list(metadata_json.get("dataLimitations"))
    ]
    metadata_analysis_rows += [
        {"category_@id": "dataCollectionType", "value": item, "value_length": len(str(item))}
        for item in as_list(metadata_json.get("dataCollectionType"))
    ]

    analysis_df = pd.DataFrame(metadata_analysis_rows)
    analysis_record_set_id = "metadata_fallback_recordset"
    numeric_field_id = "value_length"
    group_field_id = "category_@id"

    print("No record-set table was available, so a metadata-derived analysis table was created.")
    display(analysis_df.head())

In [ ]:
# Filtering, normalization, and grouping using dynamically selected fields.
record_set_id = analysis_record_set_id
numeric_field = numeric_field_id
group_field = group_field_id

if numeric_field is not None and numeric_field in analysis_df.columns:
    series = pd.to_numeric(analysis_df[numeric_field], errors="coerce")
    valid_series = series.dropna()
    
    if not valid_series.empty:
        threshold = valid_series.median()
        filtered_df = analysis_df[series > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        std_value = filtered_df[numeric_field].std() if numeric_field in filtered_df.columns else np.nan
        if pd.notnull(std_value) and std_value != 0:
            filtered_df[f"{numeric_field}_normalized"] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()
            ) / std_value
        else:
            filtered_df[f"{numeric_field}_normalized"] = 0.0

        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field is not None and group_field in filtered_df.columns:
            numeric_group_cols = filtered_df.select_dtypes(include=[np.number]).columns.tolist()
            grouped_df = filtered_df.groupby(group_field)[numeric_group_cols].mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field was available in the analysis table.")
    else:
        print("No valid numeric values were available for filtering.")
else:
    print("No numeric field was available for EDA.")

In [ ]:
# Additional descriptive summary
print(f"Analysis source @id: {record_set_id}")
print(f"Numeric field used: {numeric_field}")
print(f"Group field used: {group_field}")

display(analysis_df.describe(include="all").T)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

If tabular record data are available, the plots below use those fields directly. Otherwise, the notebook visualizes metadata-derived structures such as category counts and text-length distributions.

In [ ]:
sns.set_theme(style="whitegrid")

figures_created = 0

if numeric_field is not None and numeric_field in analysis_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(analysis_df[numeric_field], errors="coerce").dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
    figures_created += 1

if group_field is not None and group_field in analysis_df.columns:
    plt.figure(figsize=(9, 4))
    plot_counts = analysis_df[group_field].astype(str).value_counts().head(15)
    sns.barplot(x=plot_counts.values, y=plot_counts.index, orient="h")
    plt.title(f"Top categories in {group_field}")
    plt.xlabel("Count")
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()
    figures_created += 1

# If there are at least two numeric columns, create a scatterplot.
numeric_cols = analysis_df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) >= 2:
    x_col, y_col = numeric_cols[:2]
    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=analysis_df, x=x_col, y=y_col)
    plt.title(f"{y_col} vs {x_col}")
    plt.tight_layout()
    plt.show()
    figures_created += 1

if figures_created == 0:
    print("No plots were generated because the analysis table did not contain plottable fields.")

## 6. Conclusion
This notebook demonstrated how to explore a Croissant-described dataset with `mlcroissant` using a robust, schema-aware workflow.

### Key takeaways
- The dataset metadata was loaded directly from the Croissant schema URL.
- Record sets, fields, and distributions were inspected using their `@id` values when available.
- The notebook dynamically adapts to the package structure:
  - if record sets are present, it loads them into pandas DataFrames,
  - if the package is primarily metadata-oriented, it still supports analysis through metadata-derived tables.
- Basic EDA operations included filtering, normalization, grouping, descriptive summaries, and visualization.

### Notes for further work
- If future versions of the dataset expose additional `recordSet` entries, the same notebook structure can be reused without major changes.
- You can also extend this notebook to inspect individual file distributions, load external CSV/GeoPackage/GeoTIFF resources referenced by their distribution `@id`, and perform spatial analysis with tools such as `geopandas`, `rasterio`, or `xarray`.

This makes the notebook a reusable template for Croissant-based dataset exploration while remaining consistent with the requested structure and `@id`-centric referencing style.